Fix reserved keyword ROWS used as column alias in quarantine summary query
*Co-authored with CoCo*

# Police Crime Cleaning and Validation in Snowflake

**Scope:** cleaning and validation only for Metropolitan Police, West Midlands Police,
South Wales Police, and Sussex Police.

This notebook begins with a raw Snowflake table supplied by the ingestion team and
produces:

- a standardised, deduplicated clean table for downstream teammates;
- a quarantine table containing every rejected row and its rejection reason;
- run-level and force-level audit metrics;
- an explicit validation-results table.

It deliberately excludes enrichment, aggregation, trend analysis, visualisation, and
Power BI export.

## Ownership boundary

**Input contract:** one raw table containing the standard data.police.uk street-crime
columns. The raw table is not changed.

**Output contract:** record-level clean and quarantine tables. A downstream teammate
may enrich and aggregate the clean table.

## Cleaning rules

1. Trim text and turn blank strings into `NULL`.
2. Standardise the four force names and a small set of known crime-category aliases.
3. Parse `MONTH` as a first-of-month date; parse coordinates with `TRY_TO_DOUBLE`.
4. Reject rows with missing/invalid critical fields, invalid coordinates, unknown
   force/category, or missing location identifiers.
5. Deduplicate by `CRIME_ID`, retaining the most complete row deterministically.
6. Preserve all rejected rows in quarantine.

The original notebook dropped duplicate IDs with `keep=False`. Here they are handled
more conservatively: one canonical row is kept and extra copies are quarantined.

In [ ]:
from datetime import datetime, timezone
import re
import uuid

try:
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
except Exception as exc:
    raise RuntimeError(
        "Run this notebook inside Snowflake, or replace this block with a configured "
        "Snowpark Session. No credentials should be written into the notebook."
    ) from exc

RUN_ID = str(uuid.uuid4())
RUN_STARTED_AT = datetime.now(timezone.utc)

# Edit only these object names. Identifiers are validated before use.
RAW_TABLE = "CRIME_ETL_DB.RAW.STREET_CRIME"
WORK_SCHEMA = "CRIME_ETL_DB.DATA_QUALITY"
CLEAN_TABLE = f"{WORK_SCHEMA}.STREET_CRIME_CLEAN"
QUARANTINE_TABLE = f"{WORK_SCHEMA}.STREET_CRIME_QUARANTINE"
RUN_AUDIT_TABLE = f"{WORK_SCHEMA}.CLEANING_RUN_AUDIT"
FORCE_AUDIT_TABLE = f"{WORK_SCHEMA}.CLEANING_FORCE_AUDIT"
VALIDATION_TABLE = f"{WORK_SCHEMA}.VALIDATION_RESULTS"

IDENTIFIER = re.compile(r"^[A-Za-z_][A-Za-z0-9_$]*(\.[A-Za-z_][A-Za-z0-9_$]*){0,2}$")
for name in [
    RAW_TABLE, WORK_SCHEMA, CLEAN_TABLE, QUARANTINE_TABLE,
    RUN_AUDIT_TABLE, FORCE_AUDIT_TABLE, VALIDATION_TABLE
]:
    if not IDENTIFIER.fullmatch(name):
        raise ValueError(f"Unsafe Snowflake object identifier: {name}")

print(f"Run ID: {RUN_ID}")
print(f"Source: {RAW_TABLE}")

## 1. Confirm the input contract

The source table must contain these columns (case-insensitive). `SOURCE_FILE_NAME` is
recommended for lineage but is not required.

In [ ]:
REQUIRED_COLUMNS = {
    "CRIME ID", "MONTH", "REPORTED BY", "FALLS WITHIN", "LONGITUDE", "LATITUDE",
    "LOCATION", "LSOA CODE", "LSOA NAME", "CRIME TYPE", "LAST OUTCOME CATEGORY"
}

def show_column_name(row):
    values = row.as_dict()
    for key, value in values.items():
        if key.lower() in {"column_name", "name"}:
            return str(value)
    raise KeyError(f"Could not find column-name field in SHOW COLUMNS output: {values.keys()}")

source_columns = {
    show_column_name(r).upper()
    for r in session.sql(f"SHOW COLUMNS IN TABLE {RAW_TABLE}").collect()
}
missing_columns = REQUIRED_COLUMNS - source_columns
if missing_columns:
    raise ValueError(f"Source contract failed. Missing columns: {sorted(missing_columns)}")

HAS_SOURCE_FILE = "SOURCE_FILE_NAME" in source_columns
print("PASS - source contract satisfied")
print(f"SOURCE_FILE_NAME available: {HAS_SOURCE_FILE}")

## 2. Create audit structures

In [ ]:
session.sql(f"CREATE SCHEMA IF NOT EXISTS {WORK_SCHEMA}").collect()

session.sql(f"""
CREATE TABLE IF NOT EXISTS {RUN_AUDIT_TABLE} (
    RUN_ID STRING, STARTED_AT TIMESTAMP_TZ, COMPLETED_AT TIMESTAMP_TZ,
    SOURCE_TABLE STRING, CLEAN_TABLE STRING, QUARANTINE_TABLE STRING,
    RAW_TARGET_ROWS NUMBER, CLEAN_ROWS NUMBER, QUARANTINE_ROWS NUMBER,
    ROWS_REMOVED NUMBER, REMOVAL_PERCENT NUMBER(10,4), STATUS STRING
)
""").collect()

session.sql(f"""
CREATE TABLE IF NOT EXISTS {FORCE_AUDIT_TABLE} (
    RUN_ID STRING, FORCE_NAME STRING, RAW_ROWS NUMBER, CLEAN_ROWS NUMBER,
    QUARANTINE_ROWS NUMBER, DUPLICATE_ROWS NUMBER, NULL_CRIME_ID_ROWS NUMBER,
    NULL_MONTH_ROWS NUMBER, NULL_LSOA_ROWS NUMBER, NULL_COORDINATE_ROWS NUMBER
)
""").collect()

session.sql(f"""
CREATE TABLE IF NOT EXISTS {VALIDATION_TABLE} (
    RUN_ID STRING, CHECK_NAME STRING, SEVERITY STRING, OBSERVED_VALUE STRING,
    EXPECTED_VALUE STRING, PASSED BOOLEAN, CHECKED_AT TIMESTAMP_TZ
)
""").collect()
print("Audit structures ready")

## 3. Standardise and assess quality

Only the four in-scope forces are selected. The temporary stage exists only for this
Snowflake session and does not modify the raw data.

In [ ]:
source_file_expression = (
    'NULLIF(TRIM("SOURCE_FILE_NAME"), \'\')'
    if HAS_SOURCE_FILE else "CAST(NULL AS STRING)"
)

session.sql(f"""
CREATE OR REPLACE TEMPORARY TABLE STG_POLICE_CRIME_STANDARDISED AS
WITH NORMALISED AS (
    SELECT
        NULLIF(TRIM("Crime ID"), '') AS CRIME_ID,
        TRY_TO_DATE(NULLIF(TRIM("Month"), '') || '-01') AS CRIME_MONTH,
        NULLIF(TRIM("Reported by"), '') AS REPORTED_BY_RAW,
        NULLIF(TRIM("Falls within"), '') AS FALLS_WITHIN_RAW,
        TRY_TO_DOUBLE("Longitude") AS LONGITUDE,
        TRY_TO_DOUBLE("Latitude") AS LATITUDE,
        NULLIF(TRIM("Location"), '') AS LOCATION,
        UPPER(NULLIF(TRIM("LSOA code"), '')) AS LSOA_CODE,
        NULLIF(TRIM("LSOA name"), '') AS LSOA_NAME,
        NULLIF(TRIM("Crime type"), '') AS CRIME_TYPE_RAW,
        NULLIF(TRIM("Last outcome category"), '') AS LAST_OUTCOME_CATEGORY,
        {source_file_expression} AS SOURCE_FILE_NAME,
        CURRENT_TIMESTAMP() AS CLEANED_AT,
        '{RUN_ID}' AS CLEANING_RUN_ID
    FROM {RAW_TABLE}
    WHERE LOWER(TRIM("Falls within")) IN (
        'metropolitan police', 'metropolitan police service',
        'west midlands police', 'south wales police', 'sussex police'
    )
), STANDARDISED AS (
    SELECT *,
        CASE
            WHEN LOWER(FALLS_WITHIN_RAW) IN
                ('metropolitan police', 'metropolitan police service')
                THEN 'Metropolitan Police Service'
            WHEN LOWER(FALLS_WITHIN_RAW) = 'west midlands police'
                THEN 'West Midlands Police'
            WHEN LOWER(FALLS_WITHIN_RAW) = 'south wales police'
                THEN 'South Wales Police'
            WHEN LOWER(FALLS_WITHIN_RAW) = 'sussex police'
                THEN 'Sussex Police'
        END AS FORCE_NAME,
        CASE LOWER(CRIME_TYPE_RAW)
            WHEN 'anti social behaviour' THEN 'Anti-social behaviour'
            WHEN 'antisocial behaviour' THEN 'Anti-social behaviour'
            WHEN 'anti-social behaviour' THEN 'Anti-social behaviour'
            WHEN 'bicycle theft' THEN 'Bicycle theft'
            WHEN 'burglary' THEN 'Burglary'
            WHEN 'criminal damage and arson' THEN 'Criminal damage and arson'
            WHEN 'drugs' THEN 'Drugs'
            WHEN 'other crime' THEN 'Other crime'
            WHEN 'other theft' THEN 'Other theft'
            WHEN 'possession of weapons' THEN 'Possession of weapons'
            WHEN 'public order' THEN 'Public order'
            WHEN 'robbery' THEN 'Robbery'
            WHEN 'shoplifting' THEN 'Shoplifting'
            WHEN 'theft from the person' THEN 'Theft from the person'
            WHEN 'vehicle crime' THEN 'Vehicle crime'
            WHEN 'violence & sexual offences' THEN 'Violence and sexual offences'
            WHEN 'violence and sexual offences' THEN 'Violence and sexual offences'
            ELSE CRIME_TYPE_RAW
        END AS CRIME_TYPE
    FROM NORMALISED
), ASSESSED AS (
    SELECT *,
        ARRAY_TO_STRING(ARRAY_CONSTRUCT_COMPACT(
            IFF(CRIME_ID IS NULL, 'MISSING_CRIME_ID', NULL),
            IFF(CRIME_MONTH IS NULL, 'INVALID_OR_MISSING_MONTH', NULL),
            IFF(FORCE_NAME IS NULL, 'UNKNOWN_FORCE', NULL),
            IFF(CRIME_TYPE IS NULL, 'MISSING_CRIME_TYPE', NULL),
            IFF(LSOA_CODE IS NULL, 'MISSING_LSOA_CODE', NULL),
            IFF(LATITUDE IS NULL OR LONGITUDE IS NULL, 'MISSING_COORDINATE', NULL),
            IFF(LATITUDE IS NOT NULL AND NOT (LATITUDE BETWEEN 49 AND 61),
                'LATITUDE_OUT_OF_UK_RANGE', NULL),
            IFF(LONGITUDE IS NOT NULL AND NOT (LONGITUDE BETWEEN -9 AND 3),
                'LONGITUDE_OUT_OF_UK_RANGE', NULL)
        ), '|') AS BASE_REJECTION_REASON,
        ROW_NUMBER() OVER (
            PARTITION BY CRIME_ID
            ORDER BY
                IFF(CRIME_MONTH IS NOT NULL, 1, 0)
                + IFF(LSOA_CODE IS NOT NULL, 1, 0)
                + IFF(LATITUDE IS NOT NULL, 1, 0)
                + IFF(LONGITUDE IS NOT NULL, 1, 0) DESC,
                SOURCE_FILE_NAME NULLS LAST, FORCE_NAME
        ) AS CRIME_ID_RANK
    FROM STANDARDISED
)
SELECT *,
    CASE
        WHEN BASE_REJECTION_REASON <> '' AND CRIME_ID_RANK > 1
            THEN BASE_REJECTION_REASON || '|DUPLICATE_CRIME_ID'
        WHEN BASE_REJECTION_REASON <> '' THEN BASE_REJECTION_REASON
        WHEN CRIME_ID_RANK > 1 THEN 'DUPLICATE_CRIME_ID'
        ELSE NULL
    END AS REJECTION_REASON
FROM ASSESSED
""").collect()

raw_target_rows = session.table("STG_POLICE_CRIME_STANDARDISED").count()
if raw_target_rows == 0:
    raise ValueError("No rows found for the four target forces; check source table and names.")
print(f"Rows assessed: {raw_target_rows:,}")

## 4. Publish clean and quarantine tables

The `SWAP WITH` pattern replaces each published table atomically after a complete
candidate table has been built. Consumers therefore do not see a half-written result.

In [ ]:
clean_projection = """
CRIME_ID, CRIME_MONTH, YEAR(CRIME_MONTH) AS CRIME_YEAR,
MONTH(CRIME_MONTH) AS CRIME_MONTH_NUMBER, REPORTED_BY_RAW AS REPORTED_BY,
FORCE_NAME, LONGITUDE, LATITUDE, LOCATION, LSOA_CODE, LSOA_NAME, CRIME_TYPE,
LAST_OUTCOME_CATEGORY, SOURCE_FILE_NAME, CLEANED_AT, CLEANING_RUN_ID
"""

session.sql(f"""
CREATE OR REPLACE TABLE {CLEAN_TABLE}__CANDIDATE AS
SELECT {clean_projection}
FROM STG_POLICE_CRIME_STANDARDISED
WHERE REJECTION_REASON IS NULL
""").collect()

session.sql(f"""
CREATE OR REPLACE TABLE {QUARANTINE_TABLE}__CANDIDATE AS
SELECT {clean_projection}, REJECTION_REASON
FROM STG_POLICE_CRIME_STANDARDISED
WHERE REJECTION_REASON IS NOT NULL
""").collect()

def publish_candidate(target):
    exists = session.sql(
        f"SHOW TABLES LIKE '{target.split('.')[-1]}' IN SCHEMA {WORK_SCHEMA}"
    ).count() > 0
    if exists:
        session.sql(f"ALTER TABLE {target} SWAP WITH {target}__CANDIDATE").collect()
        session.sql(f"DROP TABLE {target}__CANDIDATE").collect()
    else:
        session.sql(f"ALTER TABLE {target}__CANDIDATE RENAME TO {target}").collect()

publish_candidate(CLEAN_TABLE)
publish_candidate(QUARANTINE_TABLE)
print("Clean and quarantine tables published")

## 5. Record cleaning metrics

Counts are reconciled at run and force level. Duplicate counts represent extra copies
quarantined after deterministic ranking.

In [ ]:
session.sql(f"""
INSERT INTO {FORCE_AUDIT_TABLE}
SELECT
    '{RUN_ID}', FORCE_NAME,
    COUNT(*) AS RAW_ROWS,
    COUNT_IF(REJECTION_REASON IS NULL) AS CLEAN_ROWS,
    COUNT_IF(REJECTION_REASON IS NOT NULL) AS QUARANTINE_ROWS,
    COUNT_IF(CRIME_ID_RANK > 1) AS DUPLICATE_ROWS,
    COUNT_IF(CRIME_ID IS NULL) AS NULL_CRIME_ID_ROWS,
    COUNT_IF(CRIME_MONTH IS NULL) AS NULL_MONTH_ROWS,
    COUNT_IF(LSOA_CODE IS NULL) AS NULL_LSOA_ROWS,
    COUNT_IF(LATITUDE IS NULL OR LONGITUDE IS NULL) AS NULL_COORDINATE_ROWS
FROM STG_POLICE_CRIME_STANDARDISED
GROUP BY FORCE_NAME
""").collect()

clean_rows = session.table(CLEAN_TABLE).count()
quarantine_rows = session.table(QUARANTINE_TABLE).count()
removal_pct = round(quarantine_rows / raw_target_rows * 100, 4)

session.sql(f"""
INSERT INTO {RUN_AUDIT_TABLE}
SELECT '{RUN_ID}', TO_TIMESTAMP_TZ('{RUN_STARTED_AT.isoformat()}'),
       CURRENT_TIMESTAMP(), '{RAW_TABLE}', '{CLEAN_TABLE}', '{QUARANTINE_TABLE}',
       {raw_target_rows}, {clean_rows}, {quarantine_rows}, {quarantine_rows},
       {removal_pct}, 'CLEANED_PENDING_VALIDATION'
""").collect()

session.sql(f"""
SELECT FORCE_NAME, RAW_ROWS, CLEAN_ROWS, QUARANTINE_ROWS, DUPLICATE_ROWS,
       NULL_CRIME_ID_ROWS, NULL_MONTH_ROWS, NULL_LSOA_ROWS, NULL_COORDINATE_ROWS
FROM {FORCE_AUDIT_TABLE}
WHERE RUN_ID = '{RUN_ID}'
ORDER BY FORCE_NAME
""").show()

## 6. Explicit validation suite

Critical failures stop the handoff. Warnings are recorded but do not fail the run.
Every check is written to Snowflake for auditability.

In [ ]:
EXPECTED_FORCES = {
    "Metropolitan Police Service", "West Midlands Police",
    "South Wales Police", "Sussex Police"
}

checks = []

def add_check(name, severity, observed, expected, passed):
    checks.append({
        "check_name": name, "severity": severity,
        "observed": str(observed), "expected": str(expected), "passed": bool(passed)
    })

add_check(
    "row_count_reconciliation", "CRITICAL",
    clean_rows + quarantine_rows, raw_target_rows,
    clean_rows + quarantine_rows == raw_target_rows
)
add_check("clean_table_not_empty", "CRITICAL", clean_rows, "> 0", clean_rows > 0)

actual_forces = {
    r["FORCE_NAME"]
    for r in session.sql(f"SELECT DISTINCT FORCE_NAME FROM {CLEAN_TABLE}").collect()
}
add_check(
    "four_force_coverage", "CRITICAL", sorted(actual_forces),
    sorted(EXPECTED_FORCES), actual_forces == EXPECTED_FORCES
)

critical_nulls = session.sql(f"""
SELECT COUNT(*) AS N FROM {CLEAN_TABLE}
WHERE CRIME_ID IS NULL OR CRIME_MONTH IS NULL OR FORCE_NAME IS NULL
   OR CRIME_TYPE IS NULL OR LSOA_CODE IS NULL
   OR LATITUDE IS NULL OR LONGITUDE IS NULL
""").collect()[0]["N"]
add_check("required_fields_complete", "CRITICAL", critical_nulls, 0, critical_nulls == 0)

duplicate_ids = session.sql(f"""
SELECT COUNT(*) AS N FROM (
    SELECT CRIME_ID FROM {CLEAN_TABLE}
    GROUP BY CRIME_ID HAVING COUNT(*) > 1
)
""").collect()[0]["N"]
add_check("crime_id_unique", "CRITICAL", duplicate_ids, 0, duplicate_ids == 0)

invalid_coordinates = session.sql(f"""
SELECT COUNT(*) AS N FROM {CLEAN_TABLE}
WHERE LATITUDE NOT BETWEEN 49 AND 61 OR LONGITUDE NOT BETWEEN -9 AND 3
""").collect()[0]["N"]
add_check("uk_coordinate_range", "CRITICAL", invalid_coordinates, 0, invalid_coordinates == 0)

future_months = session.sql(f"""
SELECT COUNT(*) AS N FROM {CLEAN_TABLE}
WHERE CRIME_MONTH > DATE_TRUNC('MONTH', CURRENT_DATE())
""").collect()[0]["N"]
add_check("no_future_months", "WARNING", future_months, 0, future_months == 0)

allowed_categories = {
    "Anti-social behaviour", "Bicycle theft", "Burglary", "Criminal damage and arson",
    "Drugs", "Other crime", "Other theft", "Possession of weapons",
    "Public order", "Robbery", "Shoplifting", "Theft from the person",
    "Vehicle crime", "Violence and sexual offences"
}
actual_categories = {
    r["CRIME_TYPE"]
    for r in session.sql(f"SELECT DISTINCT CRIME_TYPE FROM {CLEAN_TABLE}").collect()
}
unknown_categories = sorted(actual_categories - allowed_categories)
add_check(
    "recognised_crime_categories", "CRITICAL",
    unknown_categories, "[]", len(unknown_categories) == 0
)

for check in checks:
    observed = check["observed"].replace("'", "''")
    expected = check["expected"].replace("'", "''")
    session.sql(f"""
    INSERT INTO {VALIDATION_TABLE}
    SELECT '{RUN_ID}', '{check["check_name"]}', '{check["severity"]}',
           '{observed}', '{expected}', {str(check["passed"]).upper()}, CURRENT_TIMESTAMP()
    """).collect()

session.sql(f"""
SELECT CHECK_NAME, SEVERITY, OBSERVED_VALUE, EXPECTED_VALUE, PASSED
FROM {VALIDATION_TABLE}
WHERE RUN_ID = '{RUN_ID}'
ORDER BY IFF(SEVERITY = 'CRITICAL', 0, 1), CHECK_NAME
""").show()

critical_failures = [c for c in checks if c["severity"] == "CRITICAL" and not c["passed"]]
final_status = "VALIDATED" if not critical_failures else "VALIDATION_FAILED"
session.sql(f"""
UPDATE {RUN_AUDIT_TABLE}
SET STATUS = '{final_status}', COMPLETED_AT = CURRENT_TIMESTAMP()
WHERE RUN_ID = '{RUN_ID}'
""").collect()

if critical_failures:
    raise AssertionError(
        "Critical validation failure(s): "
        + ", ".join(c["check_name"] for c in critical_failures)
    )
print("PASS - all critical validations succeeded")

## 7. Handoff summary

Only rows from a `VALIDATED` run should be used downstream. The clean table remains at
individual-crime grain; aggregation belongs to the downstream transformation owner.

In [ ]:
session.sql(f"""
SELECT RUN_ID, STARTED_AT, COMPLETED_AT, SOURCE_TABLE, CLEAN_TABLE,
       QUARANTINE_TABLE, RAW_TARGET_ROWS, CLEAN_ROWS, QUARANTINE_ROWS,
       REMOVAL_PERCENT, STATUS
FROM {RUN_AUDIT_TABLE}
WHERE RUN_ID = '{RUN_ID}'
""").show()

session.sql(f"""
SELECT REJECTION_REASON, COUNT(*) AS ROW_COUNT
FROM {QUARANTINE_TABLE}
GROUP BY REJECTION_REASON
ORDER BY ROW_COUNT DESC
""").show()

## Output data dictionary

### Clean table

| Column | Snowflake type | Definition |
|---|---|---|
| `CRIME_ID` | STRING | Source crime identifier; unique and non-null after validation |
| `CRIME_MONTH` | DATE | Source month parsed as first day of month |
| `CRIME_YEAR` | NUMBER | Year derived from `CRIME_MONTH` |
| `CRIME_MONTH_NUMBER` | NUMBER | Month number derived from `CRIME_MONTH` |
| `REPORTED_BY` | STRING | Trimmed source reporting-force label |
| `FORCE_NAME` | STRING | Standardised in-scope force name |
| `LONGITUDE`, `LATITUDE` | FLOAT | Parsed coordinates within broad UK bounds |
| `LOCATION` | STRING | Trimmed source location text; may be null |
| `LSOA_CODE` | STRING | Upper-cased LSOA code; non-null |
| `LSOA_NAME` | STRING | Trimmed LSOA name; may be null |
| `CRIME_TYPE` | STRING | Standardised Police.uk crime category |
| `LAST_OUTCOME_CATEGORY` | STRING | Trimmed outcome; may be null because outcomes can be unavailable |
| `SOURCE_FILE_NAME` | STRING | Optional ingestion lineage |
| `CLEANED_AT` | TIMESTAMP_TZ | Cleaning timestamp |
| `CLEANING_RUN_ID` | STRING | Identifier linking rows to audit results |

### Quarantine table

Contains the same columns plus `REJECTION_REASON`, a pipe-delimited list of failed
rules. It is an exception-management dataset, not a downstream reporting source.

## Assumptions and limitations

- Input column names match the public data.police.uk street-crime extract.
- `CRIME_ID` is the record-level uniqueness key; this should be confirmed with the
  project's ingestion owner.
- Missing outcomes and location descriptions are retained because they are not
  essential to the intended reporting grain.
- LSOA and coordinate completeness are enforced to match the original notebook's
  cleaning policy and support later geographic analysis.
- Coordinate checks use broad UK bounds, not a force-boundary spatial check.
- New Police.uk crime categories will deliberately fail validation until reviewed
  and added to the approved category set.

In [ ]:
COPY INTO @CRIME_ETL_DB.CLEAN.CRIME_CLEAN_STAGE/metropolitan_police_clean.csv
FROM (
    SELECT *
    FROM CRIME_ETL_DB.DATA_QUALITY.STREET_CRIME_CLEAN
    WHERE FORCE_NAME = 'Metropolitan Police Service'
)
FILE_FORMAT = (
    TYPE = CSV
    FIELD_OPTIONALLY_ENCLOSED_BY = '"'
    COMPRESSION = NONE
    NULL_IF = ('')
)
HEADER = TRUE
SINGLE = TRUE
MAX_FILE_SIZE = 5368709120
OVERWRITE = TRUE;


COPY INTO @CRIME_ETL_DB.CLEAN.CRIME_CLEAN_STAGE/west_midlands_police_clean.csv
FROM (
    SELECT *
    FROM CRIME_ETL_DB.DATA_QUALITY.STREET_CRIME_CLEAN
    WHERE FORCE_NAME = 'West Midlands Police'
)
FILE_FORMAT = (
    TYPE = CSV
    FIELD_OPTIONALLY_ENCLOSED_BY = '"'
    COMPRESSION = NONE
    NULL_IF = ('')
)
HEADER = TRUE
SINGLE = TRUE
MAX_FILE_SIZE = 5368709120
OVERWRITE = TRUE;


COPY INTO @CRIME_ETL_DB.CLEAN.CRIME_CLEAN_STAGE/south_wales_police_clean.csv
FROM (
    SELECT *
    FROM CRIME_ETL_DB.DATA_QUALITY.STREET_CRIME_CLEAN
    WHERE FORCE_NAME = 'South Wales Police'
)
FILE_FORMAT = (
    TYPE = CSV
    FIELD_OPTIONALLY_ENCLOSED_BY = '"'
    COMPRESSION = NONE
    NULL_IF = ('')
)
HEADER = TRUE
SINGLE = TRUE
MAX_FILE_SIZE = 5368709120
OVERWRITE = TRUE;


COPY INTO @CRIME_ETL_DB.CLEAN.CRIME_CLEAN_STAGE/sussex_police_clean.csv
FROM (
    SELECT *
    FROM CRIME_ETL_DB.DATA_QUALITY.STREET_CRIME_CLEAN
    WHERE FORCE_NAME = 'Sussex Police'
)
FILE_FORMAT = (
    TYPE = CSV
    FIELD_OPTIONALLY_ENCLOSED_BY = '"'
    COMPRESSION = NONE
    NULL_IF = ('')
)
HEADER = TRUE
SINGLE = TRUE
MAX_FILE_SIZE = 5368709120
OVERWRITE = TRUE;